# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed in the current environment
!pip install mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", getattr(metadata, 'name', None))
print("Description:", getattr(metadata, 'description', None))
print("Published:", getattr(metadata, 'datePublished', None))
print("Keywords:", getattr(metadata, 'keywords', None))

## 2. Data Overview
Review record sets and their fields using their `@id` values. This is important for referencing the data elements unambiguously.

In [ ]:
# List all record sets and their @id fields
record_sets = getattr(metadata, 'recordSet', [])
if not isinstance(record_sets, list):
    record_sets = [record_sets]
    
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    rs_id = getattr(rs, "@id", None) if hasattr(rs, '@id') else rs.get("@id")
    rs_name = getattr(rs, "name", None) if hasattr(rs, 'name') else rs.get("name")
    print(f"- RecordSet @id: {rs_id}")
    print(f"  Name: {rs_name}")
    
    # List fields for this RecordSet
    fields = getattr(rs, 'field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        field_id = getattr(field, "@id", None) if hasattr(field, '@id') else field.get("@id")
        field_name = getattr(field, "name", None) if hasattr(field, 'name') else field.get("name")
        print(f"    - Field @id: {field_id}")
        print(f"      name: {field_name}")
    print()

## 3. Data Extraction
Load table data for each record set. Use the `record_set` and field `@id` values discovered above. 

Data is extracted to a pandas DataFrame for further analysis.

In [ ]:
# Collect all record set @ids for extraction
record_set_ids = []
for rs in getattr(metadata, 'recordSet', []):
    rs_id = getattr(rs, "@id", None) if hasattr(rs, '@id') else rs.get("@id")
    if rs_id:
        record_set_ids.append(rs_id)

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records from record set: {rs_id}")
    # Using .records() generator to get data
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"  -> Loaded {len(df)} rows, {len(df.columns)} columns.")
    else:
        print("  -> No records found.")
print()
# For demonstration, select the first record set
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    print(f"Columns for record set {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    dataframes[example_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Demonstrate basic filtering, normalization, and grouping using numeric and categorical fields from a chosen record set.

Replace `<numeric_field_id>` and `<group_field>` below with valid field IDs from the columns printed above.

In [ ]:
# Example: Choose a numeric field and categorical field
# Update these IDs to match the dataset's available columns
selected_record_set = example_record_set_id
df = dataframes[selected_record_set]

# You may check columns via: print(df.columns.tolist())
# For this example, let's try: 'cr:coverage_percent' for coverage and 'cr:description' for grouping if present

numeric_field = None
for candidate in ['cr:coverage_percent', 'cr:MW', 'cr:Sequence_length', 'cr:Normalized_Abundance']:
    if candidate in df.columns:
        numeric_field = candidate
        break

if numeric_field is None:
    print("No suitable numeric field found. Please inspect df.columns and select one.")
else:
    threshold = df[numeric_field].quantile(0.75) # Example: filter on top 25%
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    # Normalization
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try grouping by a categorical field
    group_field = None
    for candidate in ['cr:description', 'cr:accession', 'cr:modification']:
        if candidate in df.columns and df[candidate].dtype == object:
            group_field = candidate
            break
    print()
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped by {group_field}, mean of {numeric_field}:")
        print(grouped_df.head())
    else:
        print("No suitable categorical field found for grouping.")

## 5. Visualization
Simple visualizations for numeric and grouped data.

For more advanced visualizations, see libraries like [seaborn](https://seaborn.pydata.org/) or [plotly](https://plotly.com/python/).

In [ ]:
import matplotlib.pyplot as plt

if numeric_field and numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        grouped_plot = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        grouped_plot.plot(kind='bar')
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.tight_layout()
        plt.show()

## 6. Conclusion
This notebook demonstrates basic data exploration with `mlcroissant`, using [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json):
- Loads the dataset Croissant metadata
- Finds available record sets and fields using `@id`
- Extracts data to pandas DataFrames
- Applies basic filtering, normalization, grouping, and visualization

You can build on this notebook for more advanced statistical or machine learning workflows, or integrate it as part of a reproducible data analysis pipeline.

For additional resources, see:
- [mlcroissant documentation](https://mlcommons.github.io/croissant/)
- [SenScience FAIR^2 platform](https://sen.science/)
